# GBWM + RL Capstone â€” Phase 1

**Run order:** Cell 1 (install) â†’ Kernel Restart â†’ Run All

In [1]:
import subprocess, sys

packages = ['stable-baselines3', 'gymnasium', 'numpy', 'pandas', 'scipy', 'matplotlib']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    print(f'  âœ“  {pkg}')

print('\nâœ“ Packages installed')
print('âš   Now go to: Kernel â†’ Restart & Run All')


  âœ“  stable-baselines3
  âœ“  gymnasium
  âœ“  numpy
  âœ“  pandas
  âœ“  scipy
  âœ“  matplotlib

âœ“ Packages installed
âš   Now go to: Kernel â†’ Restart & Run All


# GBWM Environment

In [2]:
"""
gbwm_env.py
============
Custom GBWM environment implementing:
  - Geometric Brownian Motion wealth evolution (Eq. 1, Das et al. 2024)
  - 15-portfolio discrete action space (Section III-B)
  - State space: (normalized_wealth, time_remaining)
  - Sparse reward: +1 if W(T) >= G, else 0

Paper reference: Das, Mittal, Ostrov et al. (2024)
                 "RL for Multiple Goals in GBWM"
"""

import numpy as np
from dataclasses import dataclass, field
from typing import Optional


# â”€â”€ Paper Table 1 parameters â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
MU_ANNUAL  = np.array([0.0493, 0.0770, 0.0886])
COV_ANNUAL = np.array([
    [ 0.0017, -0.0017, -0.0021],
    [-0.0017,  0.0396,  0.0309],
    [-0.0021,  0.0309,  0.0392],
])
MU_MIN, MU_MAX = 0.0526, 0.0886
N_PORTFOLIOS   = 15


# â”€â”€ Build 15 frontier portfolios â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def _build_frontier():
    Si   = np.linalg.inv(COV_ANNUAL)
    ones = np.ones(3)
    k = MU_ANNUAL @ Si @ ones
    l = MU_ANNUAL @ Si @ MU_ANNUAL
    p = ones       @ Si @ ones
    D = l*p - k**2
    g = (l * Si @ ones       - k * Si @ MU_ANNUAL) / D
    h = (p * Si @ MU_ANNUAL  - k * Si @ ones)      / D
    a = float(h @ COV_ANNUAL @ h)
    b = float(2*(g @ COV_ANNUAL @ h))
    c = float(g @ COV_ANNUAL @ g)

    mu_grid  = np.linspace(MU_MIN, MU_MAX, N_PORTFOLIOS)
    sig_grid = np.sqrt(np.maximum(a*mu_grid**2 + b*mu_grid + c, 0))
    return mu_grid, sig_grid

PORTFOLIO_MU, PORTFOLIO_SIGMA = _build_frontier()


# â”€â”€ Investor scenarios â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
@dataclass
class InvestorProfile:
    name:       str
    W0:         float
    G:          float
    T:          int
    cash_flows: np.ndarray = field(default_factory=lambda: np.zeros(0))

    def __post_init__(self):
        if len(self.cash_flows) == 0:
            self.cash_flows = np.zeros(self.T)


SCENARIOS = {
    "base": InvestorProfile(
        name="Base case",
        W0=100_000, G=200_000, T=10,
        cash_flows=np.zeros(10),
    ),
    "young": InvestorProfile(
        name="Young investor",
        W0=50_000, G=500_000, T=30,
        cash_flows=np.full(30, 5_000),
    ),
    "retirement": InvestorProfile(
        name="Retirement",
        W0=100_000, G=121_400, T=30,
        cash_flows=np.array(
            [15_000]*15 + [-50_000*(1.03**t) for t in range(1,16)]
        ),
    ),
    "stressed": InvestorProfile(
        name="Stressed",
        W0=100_000, G=200_000, T=10,
        cash_flows=np.full(10, -10_000),
    ),
}


# â”€â”€ GBM Environment â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
class GBWMEnv:
    """
    GBWM environment following Das et al. 2024 Section II-B.

    State:   s = [W(t)/G,  (T-t)/T]   â€” 2D continuous
    Action:  a âˆˆ {0,1,...,14}          â€” portfolio index
    Reward:  0 at each step; +1 at t=T if W(T) >= G
    """

    def __init__(self, profile: InvestorProfile, seed: Optional[int] = None):
        self.profile = profile
        self.rng     = np.random.default_rng(seed)
        self.W       = 0.0
        self.t       = 0

    # â”€â”€ reset â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    def reset(self) -> np.ndarray:
        self.W = self.profile.W0
        self.t = 0
        return self._state()

    # â”€â”€ step â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    def step(self, action: int):
        """
        One year of GBM evolution.
        Implements Eq. 1: W(t+1) = (W(t)+C(t)) * exp((Âµ-ÏƒÂ²/2) + ÏƒZ)
        """
        assert 0 <= action < N_PORTFOLIOS, f"Invalid action {action}"

        mu  = PORTFOLIO_MU[action]
        sig = PORTFOLIO_SIGMA[action]
        cf  = self.profile.cash_flows[self.t]

        # Apply cash flow
        W_pre = self.W + cf

        # Bankruptcy check
        if W_pre <= 0:
            self.W = 0.0
            self.t += 1
            done   = self.t >= self.profile.T
            reward = 1.0 if (done and self.W >= self.profile.G) else 0.0
            return self._state(), reward, done

        # GBM step (Eq. 1)
        Z      = self.rng.standard_normal()
        self.W = W_pre * np.exp((mu - 0.5*sig**2) + sig*Z)
        self.t += 1

        done   = self.t >= self.profile.T
        reward = 1.0 if (done and self.W >= self.profile.G) else 0.0
        return self._state(), reward, done

    # â”€â”€ internal â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    def _state(self) -> np.ndarray:
        G = self.profile.G
        T = self.profile.T
        w_norm = np.clip(self.W / G, 0, 5)
        t_norm = (T - self.t) / T
        return np.array([w_norm, t_norm], dtype=np.float32)


# â”€â”€ Simulation helpers â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def simulate_policy(profile: InvestorProfile,
                    policy_fn,
                    n_episodes: int = 10_000,
                    seed: int = 0) -> dict:
    """
    Run n_episodes under a given policy and return statistics.
    policy_fn: callable(state) -> action (int)
    """
    env      = GBWMEnv(profile, seed=seed)
    terminal = np.zeros(n_episodes)

    for ep in range(n_episodes):
        state = env.reset()
        env.rng = np.random.default_rng(seed + ep)
        done  = False
        while not done:
            action        = policy_fn(state)
            state, r, done = env.step(action)
        terminal[ep] = env.W

    success = terminal >= profile.G
    return {
        "pr_goal":        float(success.mean()),
        "pr_bankruptcy":  float((terminal <= 0).mean()),
        "mean_terminal":  float(terminal.mean()),
        "p10":            float(np.percentile(terminal, 10)),
        "median":         float(np.median(terminal)),
        "p90":            float(np.percentile(terminal, 90)),
        "terminal":       terminal,
    }


# â”€â”€ Fixed benchmark policies â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def tdf_policy(state) -> int:
    """Target Date Fund: linearly glide from P12 â†’ P2 as time runs out."""
    t_remaining = state[1]          # (T-t)/T
    port = int(round(2 + t_remaining * 10))
    return int(np.clip(port, 0, N_PORTFOLIOS-1))

def aggressive_policy(state) -> int:
    return N_PORTFOLIOS - 1         # always P14

def conservative_policy(state) -> int:
    return 0                        # always P0

def moderate_policy(state) -> int:
    return N_PORTFOLIOS // 2        # always P7

def greedy_policy(state) -> int:
    """High-risk when behind goal, low-risk when ahead â€” heuristic."""
    w_norm = state[0]
    if w_norm < 0.7:   return 12
    elif w_norm < 1.0: return 7
    else:              return 2


# GBWMGymEnv â€” Gymnasium wrapper (required for Stable-Baselines3)

In [3]:
import gymnasium as gym
from gymnasium import spaces

class GBWMGymEnv(gym.Env):
    def __init__(self, profile):
        super().__init__()
        self.profile = profile
        self.W = profile.W0
        self.t = 0
        self.rng = np.random.default_rng()
        self.observation_space = spaces.Box(
            low  = np.array([0., 0.], dtype=np.float32),
            high = np.array([5., 1.], dtype=np.float32))
        self.action_space = spaces.Discrete(N_PORTFOLIOS)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        self.W = self.profile.W0
        self.t = 0
        return self._obs(), {}

    def step(self, action):
        mu  = PORTFOLIO_MU[int(action)]
        sig = PORTFOLIO_SIGMA[int(action)]
        cf  = (self.profile.cash_flows[self.t]
               if self.t < len(self.profile.cash_flows) else 0.)
        W_pre = self.W + cf
        if W_pre <= 0:
            self.W = 0.
            self.t += 1
            done = self.t >= self.profile.T
            return self._obs(), 0., done, False, {}
        Z = self.rng.standard_normal()
        self.W = W_pre * np.exp((mu - 0.5*sig**2) + sig*Z)
        self.t += 1
        done = self.t >= self.profile.T
        if done:
            reward = 1.0 if self.W >= self.profile.G else 0.0
        else:
            # Reward shaping: stronger intermediate signal for stressed scenario
            w_ratio = float(np.clip(self.W / self.profile.G, 0, 1))
            reward = 0.01 * w_ratio          # 10x stronger baseline
            if w_ratio >= 0.5:               # bonus for staying above half-goal
                reward += 0.005
        return self._obs(), reward, done, False, {}

    def _obs(self):
        return np.array([
            float(np.clip(self.W / self.profile.G, 0, 5)),
            float((self.profile.T - self.t) / self.profile.T)
        ], dtype=np.float32)




# PPO Agent (from scratch â€” reference implementation)

In [4]:
"""
ppo_agent.py
=============
Proximal Policy Optimization (PPO) implemented from scratch with NumPy.
Matches the architecture in Das, Mittal, Ostrov et al. (2024) Section II-C:
  - 2 inputs  (W/G, time_remaining)
  - 2 hidden layers of N_neur neurons (default 64)
  - 15 outputs (softmax policy over portfolios)
  - Separate value network with same shape
  - Clip parameter Îµ = 0.50
  - Learning rate Î· = 0.01 (linear decay to 0)
  - Base case: N_traj=50,000 Â· batch M=4,800
"""

import numpy as np


# â”€â”€ Neural network utilities â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def relu(x):        return np.maximum(0, x)
def relu_grad(x):   return (x > 0).astype(float)
def softmax(x):
    x = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)


class MLP:
    """
    Simple fully-connected network.
    Shape: input_dim â†’ n_hidden â†’ n_hidden â†’ output_dim
    """
    def __init__(self, input_dim, hidden_dim, output_dim, seed=0):
        rng   = np.random.default_rng(seed)
        scale = lambda i, o: np.sqrt(2.0 / i)

        self.W1 = rng.normal(0, scale(input_dim,  hidden_dim), (input_dim,  hidden_dim))
        self.b1 = np.zeros(hidden_dim)
        self.W2 = rng.normal(0, scale(hidden_dim, hidden_dim), (hidden_dim, hidden_dim))
        self.b2 = np.zeros(hidden_dim)
        self.W3 = rng.normal(0, scale(hidden_dim, output_dim), (hidden_dim, output_dim))
        self.b3 = np.zeros(output_dim)

    def forward(self, x):
        h1 = relu(x  @ self.W1 + self.b1)
        h2 = relu(h1 @ self.W2 + self.b2)
        out = h2 @ self.W3 + self.b3
        return out, (x, h1, h2)

    def params(self):
        return [self.W1, self.b1, self.W2, self.b2, self.W3, self.b3]

    def apply_grads(self, grads, lr):
        for p, g in zip(self.params(), grads):
            p -= lr * g

    def backward_value(self, cache, delta):
        x, h1, h2 = cache
        dh2  = delta @ self.W3.T * relu_grad(h2)
        dh1  = dh2   @ self.W2.T * relu_grad(h1)
        return [
            x.T  @ dh1, dh1.sum(0),
            h1.T @ dh2, dh2.sum(0),
            h2.T @ delta, delta.sum(0),
        ]


# â”€â”€ PPO Agent â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
class PPOAgent:
    """
    PPO agent for GBWM portfolio selection.

    Hyperparameters match paper base case (Section II-F):
      N_traj = 50,000   total trajectories
      M      = 4,800    batch size
      Î·      = 0.01     initial learning rate
      Îµ      = 0.50     clip parameter
      N_neur = 64       neurons per hidden layer
    """

    def __init__(
        self,
        n_actions:   int   = 15,
        n_hidden:    int   = 64,
        lr:          float = 0.01,
        clip_eps:    float = 0.50,
        gamma:       float = 1.0,    # no discounting (sparse terminal reward)
        seed:        int   = 0,
    ):
        self.n_actions = n_actions
        self.lr0       = lr
        self.lr        = lr
        self.clip_eps  = clip_eps
        self.gamma     = gamma

        # Policy network: outputs logits â†’ softmax
        self.policy_net = MLP(2, n_hidden, n_actions, seed=seed)
        # Value network: outputs scalar estimate
        self.value_net  = MLP(2, n_hidden, 1, seed=seed+1)

        self.total_steps  = 0
        self.total_traj   = 0
        self.train_log    = []          # (traj_num, mean_reward)

    # â”€â”€ Action selection â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    def get_action(self, state: np.ndarray, deterministic: bool = False) -> tuple:
        """Returns (action, log_prob)."""
        logits, _ = self.policy_net.forward(state[None])
        probs      = softmax(logits)[0]

        if deterministic:
            action = int(np.argmax(probs))
        else:
            action = int(np.random.choice(self.n_actions, p=probs))

        log_prob = np.log(probs[action] + 1e-8)
        return action, log_prob

    def policy_fn(self, state: np.ndarray) -> int:
        """Deterministic policy for evaluation."""
        action, _ = self.get_action(state, deterministic=True)
        return action

    def get_value(self, state: np.ndarray) -> float:
        val, _ = self.value_net.forward(state[None])
        return float(val[0, 0])

    # â”€â”€ Collect one episode â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    def collect_episode(self, env) -> dict:
        states, actions, log_probs, rewards = [], [], [], []
        state = env.reset()
        done  = False

        while not done:
            action, lp         = self.get_action(state)
            next_state, r, done = env.step(action)
            states.append(state)
            actions.append(action)
            log_probs.append(lp)
            rewards.append(r)
            state = next_state

        # Compute returns (sparse: only terminal reward)
        T       = len(rewards)
        returns = np.zeros(T)
        G       = 0.0
        for i in reversed(range(T)):
            G         = rewards[i] + self.gamma * G
            returns[i] = G

        return {
            "states":    np.array(states,    dtype=np.float32),
            "actions":   np.array(actions,   dtype=np.int32),
            "log_probs": np.array(log_probs, dtype=np.float32),
            "returns":   returns.astype(np.float32),
            "reward":    float(sum(rewards)),
        }

    # â”€â”€ PPO update on a batch â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    def update(self, batch: list, n_epochs: int = 4):
        """
        Implements PPO clipped surrogate objective.
        batch: list of episode dicts from collect_episode()
        """
        # Concatenate batch
        states    = np.vstack([e["states"]    for e in batch])
        actions   = np.concatenate([e["actions"]   for e in batch])
        old_lps   = np.concatenate([e["log_probs"] for e in batch])
        returns   = np.concatenate([e["returns"]   for e in batch])

        # Normalize advantages
        values, _ = self.value_net.forward(states)
        values    = values[:, 0]
        advs      = returns - values
        if advs.std() > 1e-8:
            advs = (advs - advs.mean()) / (advs.std() + 1e-8)

        for _ in range(n_epochs):
            # â”€â”€ Policy loss (clipped surrogate) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
            logits, cache_p = self.policy_net.forward(states)
            probs  = softmax(logits)
            new_lps = np.log(probs[np.arange(len(actions)), actions] + 1e-8)

            ratio   = np.exp(new_lps - old_lps)
            clip_r  = np.clip(ratio, 1 - self.clip_eps, 1 + self.clip_eps)
            pol_loss = -np.minimum(ratio * advs, clip_r * advs).mean()

            # Gradient of policy loss w.r.t. logits
            N       = len(states)
            d_logits = probs.copy()
            pg      = -(np.minimum(ratio, clip_r) * advs)[:, None]
            d_logits[np.arange(N), actions] -= 1
            d_logits = d_logits * pg / N

            grads_p = self.policy_net.backward_value(cache_p, d_logits)
            self.policy_net.apply_grads(grads_p, self.lr)

            # â”€â”€ Value loss (MSE) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
            vals, cache_v = self.value_net.forward(states)
            val_loss      = 0.5 * ((vals[:, 0] - returns)**2).mean()
            d_val         = ((vals[:, 0] - returns) / N)[:, None]
            grads_v       = self.value_net.backward_value(cache_v, d_val)
            self.value_net.apply_grads(grads_v, self.lr)

    # â”€â”€ Training loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    def train(
        self,
        env_factory,          # callable() â†’ GBWMEnv fresh instance
        n_traj:     int = 50_000,
        batch_size: int = 4_800,
        log_every:  int = 500,
        verbose:    bool = True,
    ) -> list:
        """
        Main training loop following Figure 1 of Das et al. 2024.
        Returns list of (traj_num, mean_batch_reward).
        """
        log   = []
        batch = []
        traj  = 0

        while traj < n_traj:
            # Linear LR decay
            self.lr = self.lr0 * (1 - traj / n_traj)

            env = env_factory()
            ep  = self.collect_episode(env)
            batch.append(ep)
            traj += 1

            if len(batch) >= batch_size or traj >= n_traj:
                self.update(batch)
                mean_r = np.mean([e["reward"] for e in batch])
                log.append((traj, mean_r))

                if verbose and traj % log_every < batch_size:
                    print(f"  traj {traj:>6}/{n_traj}  |  "
                          f"mean reward = {mean_r:.4f}  |  "
                          f"lr = {self.lr:.5f}")
                batch = []

        self.train_log = log
        self.total_traj = traj
        return log


# Train & Evaluate

In [5]:
import numpy as np
import time, json, os
from stable_baselines3 import PPO as SB3_PPO

np.random.seed(42)

RESULTS_DIR = 'results'
N_TEST      = 10_000
N_TRAJ      = 50_000
BATCH_SIZE  = 4_800

print('=' * 60)
print('GBWM PPO TRAINING & EVALUATION')
print('Paper: Das, Mittal, Ostrov et al. (2024)')
print('=' * 60)

all_results = {}

for scenario_key, profile in SCENARIOS.items():
    print(f"\n" + '-'*60)
    print(f'SCENARIO: {profile.name}')
    print(f'  W0=${profile.W0:,.0f}  G=${profile.G:,.0f}  T={profile.T}yr')
    print('-'*60)

    print('\n[1/3] Training PPO agent (Stable-Baselines3)...')
    t0  = time.time()
    env = GBWMGymEnv(profile)

    model = SB3_PPO(
        'MlpPolicy', env,
        learning_rate = 0.01,
        n_steps       = BATCH_SIZE,
        clip_range    = 0.50,
        policy_kwargs = dict(net_arch=[64, 64]),
        verbose       = 0,
        seed          = 42,
    )
    model.learn(total_timesteps=N_TRAJ)
    training_log = []
    train_time = time.time() - t0
    print(f'  Training time: {train_time:.1f}s')

    def ppo_fn(state):
        obs = np.array(state, dtype=np.float32).reshape(1, -1)
        action, _ = model.predict(obs, deterministic=True)
        return int(action)

    print(f'\n[2/3] Evaluating strategies ({N_TEST:,} episodes each)...')
    strategies = {
        'PPO (RL)':      ppo_fn,
        'TDF':           tdf_policy,
        'Aggressive':    aggressive_policy,
        'Conservative':  conservative_policy,
        'Moderate':      moderate_policy,
        'Greedy':        greedy_policy,
    }

    eval_results = {}
    for name, fn in strategies.items():
        res = simulate_policy(profile, fn, n_episodes=N_TEST, seed=999)
        eval_results[name] = res
        print(f'  {name:<18} Pr[goal]={res["pr_goal"]:.4f}  median=${res["median"]:>10,.0f}')

    dp_approx = {'base':0.669,'young':0.750,'retirement':0.586,'stressed':0.380}
    dp_pr  = dp_approx.get(scenario_key, 0.65)
    ppo_pr = eval_results['PPO (RL)']['pr_goal']
    rl_eff = ppo_pr / dp_pr if dp_pr > 0 else 0.0

    print(f'\n  DP benchmark (from paper): {dp_pr:.4f}')
    print(f'  PPO result:                {ppo_pr:.4f}')
    print(f'  RL Efficiency:             {rl_eff:.4f}  ({rl_eff*100:.1f}% of DP)')

    all_results[scenario_key] = {
        'profile':        {'name':profile.name,'W0':profile.W0,'G':profile.G,'T':profile.T},
        'strategies':     {k:{kk:vv for kk,vv in v.items() if kk!='terminal'} for k,v in eval_results.items()},
        'dp_pr_goal':     dp_pr,
        'ppo_pr_goal':    ppo_pr,
        'rl_efficiency':  rl_eff,
        'training_log':   training_log,
        'train_time_s':   train_time,
        'terminal_wealth':{k:v['terminal'].tolist() for k,v in eval_results.items()},
    }

print('\n[3/3] Saving results...')
os.makedirs(RESULTS_DIR, exist_ok=True)
with open(f'{RESULTS_DIR}/all_results.json','w') as f:
    json.dump(all_results, f, indent=2)

print('\n' + '='*65)
print(f'{"SCENARIO":<20} {"PPO":>8} {"DP":>8} {"TDF":>8} {"RL Eff":>10}')
print('='*65)
for sk, res in all_results.items():
    ppo=res['ppo_pr_goal']; dp=res['dp_pr_goal']
    tdf=res['strategies']['TDF']['pr_goal']; eff=res['rl_efficiency']
    flag='âœ“' if eff>=0.90 else ('âš ' if eff>=0.70 else 'âœ—')
    print(f"{res['profile']['name']:<20} {ppo:>8.4f} {dp:>8.4f} {tdf:>8.4f} {eff:>9.1%}  {flag}")
print('='*65)
print(f'\nâœ“ Results saved to {RESULTS_DIR}/all_results.json')


GBWM PPO TRAINING & EVALUATION
Paper: Das, Mittal, Ostrov et al. (2024)

------------------------------------------------------------
SCENARIO: Base case
  W0=$100,000  G=$200,000  T=10yr
------------------------------------------------------------

[1/3] Training PPO agent (Stable-Baselines3)...
  Training time: 44.9s

[2/3] Evaluating strategies (10,000 episodes each)...


C:\Users\Owner\AppData\Local\Temp\ipykernel_10764\356909202.py:46: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(action)


  PPO (RL)           Pr[goal]=0.6137  median=$   215,747
  TDF                Pr[goal]=0.4575  median=$   192,500
  Aggressive         Pr[goal]=0.5104  median=$   202,531
  Conservative       Pr[goal]=0.0712  median=$   168,406
  Moderate           Pr[goal]=0.4571  median=$   193,212
  Greedy             Pr[goal]=0.5728  median=$   216,388

  DP benchmark (from paper): 0.6690
  PPO result:                0.6137
  RL Efficiency:             0.9173  (91.7% of DP)

------------------------------------------------------------
SCENARIO: Young investor
  W0=$50,000  G=$500,000  T=30yr
------------------------------------------------------------

[1/3] Training PPO agent (Stable-Baselines3)...
  Training time: 41.1s

[2/3] Evaluating strategies (10,000 episodes each)...
  PPO (RL)           Pr[goal]=0.9067  median=$   795,645
  TDF                Pr[goal]=0.8792  median=$   828,880
  Aggressive         Pr[goal]=0.7817  median=$   969,143
  Conservative       Pr[goal]=0.8897  median=$   608,76